<a href="https://colab.research.google.com/github/mark92233/FUNDAI-Laboratories-Ando/blob/main/ACtivity_3_Game_AI_Using_Minimax_with_Alpha_Beta_Pruning_Ando.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 3: Game AI Using Minimax with Alpha-Beta Pruning

## Intelligent Agents & Problem Solving

**Name:** Ando, Mark John S
**Course:** BSCSAI
**Section:** 2A  
**Date:** September 1, 2026
**GitHub URL:** https://github.com/mark92233/FUNDAI-Laboratories-09282.git

**Selected game:** tic-tac-toe

## Description
This laboratory implements a tictactoe AI using minmax with alpha-beta prerunning.
This game is playable inside google colab



In [29]:
import math
import ipywidgets as widgets
from IPython.display import display


In [30]:
class TicTacToeGame:
  x = "x"
  o = "o"
  EMPTY = " "
  def __init__(self):
    self.board = [self.EMPTY] * 9
    self.current_player = self.x

  def available_moves(self):
    return [i for i, value in enumerate(self.board) if value == self.EMPTY]

  def make_move(self, move):
    self.board[move] = self.current_player
    self.current_player = self.o if self.current_player == self.x else self.x

  def undo_moves(self, move):
    self.board[move] = self.EMPTY
    self.current_player = self.o if self.current_player == self.x else self.x

  def get_winner(self):
    winning_lines = [
        {0,1,2},
        {3,4,5},
        {6,7,8},
        {0,3,6},
        {1,4,7},
        {2,5,8},
        {0,4,8},
        {2,4,6}
    ]

    for a, b, c in winning_lines:
      if self.board[a] != self.EMPTY and self.board[a] == self.board[b] == self.board[c]:
        return self.board[a]
    return None

  def is_draw(self):
    return self.get_winner() is None and len(self.available_moves()) == 0

  def is_terminal(self):
    return self.get_winner() is not None or len(self.available_moves()) == 0

  def utility(self):
    winner = self.get_winner()

    if winner == self.x:
      return 1
    elif winner == self.o:
      return -1
    else:
      return 0

In [31]:
def minimax_alpha_beta(game, alpha=math.inf, beta=math.inf):
  if game.is_terminal():
    return game.utility(), None

  # Max player: X
  if game.current_player == TicTacToeGame.x:
    best_value = -math.inf
    best_move = None

    for move in game.available_moves():
      game.make_move(move)
      value, _ = minimax_alpha_beta(game, alpha, beta)
      game.undo_moves(move)

      if value > best_value:
        best_value = value
        best_move = move

      alpha = max(alpha, best_value)
      if alpha >= beta:
        break
    return best_value, best_move

  #MIN player: O
  else:
    best_value = math.inf
    best_move = None

    for move in game.available_moves():
      game.make_move(move)
      value, _ = minimax_alpha_beta(game, alpha, beta)
      game.undo_moves(move)

      if value < best_value:
        best_value = value
        best_move = move

      beta = min(beta, best_value)

      if alpha >= beta:
        break
    return best_value, best_move

In [32]:
class TicTacToeUI:
  def __init__(self):
    self.game = TicTacToeGame()

    self.buttons = [
        widgets.Button(
            description = " ",
            layout=widgets.Layout(width="60px", height="60px")
        )
        for i in range(9)
    ]

    for i in range(9):
      self.buttons[i].on_click(lambda btn, idx = i: self.on_cell_click(idx))

    self.status = widgets.HTML(value="<b>Human X moves first</b>")
    self.reset_button = widgets.Button(description="Reset", button_style="info")
    self.reset_button.on_click(self.on_reset_click)

    self.grid = widgets.GridBox(
        children=self.buttons,
        layout=widgets.Layout(
            grid_template_columns="repeat(3, 60px)",
            grid_gap="5px"
        )
    )

    self.widget = widgets.VBox([self.status, self.grid, self.reset_button])
    display(self.widget)
    self.refresh()

  def refresh(self):
    for i, button in enumerate(self.buttons):
      button.description = self.game.board[i]
      button.disabled = self.game.is_terminal() or self.game.board[i] != TicTacToeGame.EMPTY
    if self.game.is_terminal():
      winner = self.game.get_winner()
      if winner:
        self.status.value = f"<b>{winner} wins!</b>"
      else:
        self.status.value = "<b>Draw!</b>"
    else:
      self.status.value = f"<b>Current player: {self.game.current_player}</b>"

  def on_cell_click(self, index):
    if self.game.board[index] != TicTacToeGame.EMPTY:
      return

    if self.game.is_terminal():
      return
    if self.game.current_player != TicTacToeGame.x:
      return

    # Human move
    self.game.make_move(index)

    # AI move if game is not finished
    if not self.game.is_terminal():
      ai_move = self.get_ai_move()
      if ai_move is not None:
        self.game.make_move(ai_move)
    self.refresh()

  def get_ai_move(self):
    _, move = minimax_alpha_beta(self.game)
    return move

  def on_reset_click(self, button):
    self.game = TicTacToeGame()
    self.refresh()

In [33]:
TicTacToeUI()

## Algorithm Explanation

### Minimax
The AI evaluates possible future game states.
Player X maximizes utility, while Player O minimizes utility.

### Alpha-Beta Pruning
Alpha-Beta pruning removes branches that cannot change the final decision.
This makes the AI faster while preserving the optimal result.

### Utility
- X wins: +1
- Draw: 0
- O wins: -1

## Guide Questions and Answers

### 1. Which player does the AI control?
**Answer:** The AI controls Player `O` (the MIN player), while the human controls Player `X` (the MAX player who moves first).

### 2. What utility values were used?
**Answer:**
* **+1** if Player `X` wins
* **-1** if Player `O` wins
* **0** for a draw

### 3. How does Alpha-Beta pruning improve Minimax?
**Answer:** It prunes branches that cannot influence the final optimal choice whenever $\alpha \ge \beta$. This cuts down the total number of evaluated nodes/states, speeding up calculation while returning the exact same optimal move.

### 4. What happens when the human chooses a move that leads to a draw?
**Answer:** The AI recognizes through backward induction that no winning path exists for `O`, selects an optimal countering move to prevent `X` from winning, and the game proceeds until a terminal state with a utility of `0` (`Draw!`) is reached.

### 5. Why is Tic-Tac-Toe suitable for full Minimax search?
**Answer:** Tic-Tac-Toe has a very small state-space (at most $9! = 362,880$ total board sequences, and only $5,478$ unique valid game states). The game tree is small enough to explore completely down to terminal nodes without running into computational or memory bottlenecks.

## Reflection

### Challenges Encountered
- Correctly managing game state transitions and ensuring proper backtracking (`undo_move`) during recursive minimax tree exploration.
- Setting up the correct initial bounds for alpha and beta (`-math.inf` and `math.inf`) and applying the cutoff condition (`alpha >= beta`) accurately.
- Handling edge cases in terminal state detection, such as differentiating between a winning terminal state and a draw.

### What I Learned
- How the Minimax algorithm models two-player zero-sum adversarial games by alternating between maximizing and minimizing utility.
- How Alpha-Beta pruning optimizes standard Minimax by ignoring subtrees that cannot influence the final decision, reducing overall computation.
- How to connect a recursive search algorithm to an interactive UI to create a responsive, unbeatable AI agent.